In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("..//data//anthropometric_2003_2023.csv")
df.head(3)

,Year,Gender,Age,1.Body Weight (kg),2.Stature height (cm),3.The height of the root of the nose in standing (cm),4.Height of shoulders in standing position (cm),5.Height of the elbow in standing position (cm),6.The height of the tip of the 3rd finger in standing position (cm),7.Arm's reach in standing position (cm),...,16.Knee height in sitting position (cm),17.Arm's reach in sitting position (cm),18.The length of the forearm and hand at the elbow bend (cm),19.Thigh length in sitting position at knee bend (cm),20.Leg length when sitting forward (cm),21.Arm's reach in sitting position (cm),22.Palm width (cm),23.Palm length (cm),24.Foot width (cm),25.Foot lenght (cm)
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2003.0,M,18,65,"173,5","168,0","144,0","105,0","67,0","220,0",...,"54,0","134,0","48,0","60,0","102,0","88,0","11,0","19,0","10,0","25,0"
2,2003.0,M,18,73,"170,0","168,0","152,0","118,0","71,0","224,0",...,"55,0","138,0","50,0","54,0","106,0","74,0","8,0","19,0","11,0","22,0"


In [42]:
# --- Limpieza de datos

# Eliminar columnas
def drop_columns(data: pd.DataFrame, column_names: list) -> pd.DataFrame:
    return data[column_names]

# Eliminar nulos
def drop_nulls(data: pd.DataFrame) -> pd.DataFrame:
    return data.dropna()

# Resumen
def check_quality(data: pd.DataFrame):

    """Basic Quality Check. Returns DataFrame with columns, types 
    and unique, missing and duplicate elements plus their ratios.
    Complement/extension of .info() function."""

    n = len(data)

    summary = []
    for column in data:
        col_type = type(data[column].iloc[0])
        elements = data[column].count()
        unique = data[column].nunique(dropna = True)
        rate_unique = round(unique / n * 100, 2)
        missing = data[column].isna().sum() 
        rate_missing = round(missing / n * 100, 2)
        duplicated = data[column].duplicated().sum()
        
        summary.append((column, col_type, elements, unique, rate_unique, missing, rate_missing, duplicated))
    
    return pd.DataFrame(summary, columns = ['column', 'type', 'elements', 'unique', 'rate_unique', 'missing', 'rate_missing', 'duplicated'])


# Cambiar type
def change_type(data: pd.DataFrame, change_dict: dict) -> pd.DataFrame:

    """Changes column types to correct type indicated on dictionary values."""

    for column, correct_type in change_dict.items():
        
        if column not in data.columns:
            raise ValueError(f"Column {column} does not exist.")
        
        if change_dict[column] == 'float':
            data[column] = data[column].astype(str).str.replace(r',', '.').str.replace(' kg', '').str.replace(' cm', '')
        
        data[column] = data[column].astype(correct_type)

    return data
    
def clean_df(data):

    """Cleans data by:
    * Selecting only useful columns.
    * Dropping rows with null values.
    * Renaming columns.
    * Changing column to correct types.
    Returns clean data."""

    # Choosing only useful columns
    df_red = drop_columns(data, ['Year', 'Gender', 'Age', '1.Body Weight (kg)', '2.Stature height (cm)',
                           '3.The height of the root of the nose in standing (cm)', '4.Height of shoulders in standing position (cm)',
                           '5.Height of the elbow in standing position (cm)', '6.The height of the tip of the 3rd finger in standing position (cm)',
                           '8.The width of the shoulders (cm)', "10.Arm's reach in standing position (cm)", '16.Knee height in sitting position (cm)',
                           '18.The length of the forearm and hand at the elbow bend  (cm)', '20.Leg length when sitting forward (cm)',
                           '22.Palm width (cm)', '23.Palm length (cm)', '24.Foot width (cm)', '25.Foot lenght (cm)'])

    # Dropping nulls
    df_red = drop_nulls(df_red)

    # Renaming columns
    df_red.columns = ['year', 'gender', 'age', 'weight', 'height', 'eye_height', 'shoulder_height', 'elbow_height', 'fingertip_height', 'shoulder_width', 'arms_reach', 'knee_height', 'fingertip_to_elbow_length', 'leg_length', 'palm_width', 'palm_length', 'foot_width', 'foot_length']

    # Changing column type
    change_dict = {}
    for column in df_red.columns:
        if column != 'gender':
            change_dict[column] = 'float'
        else:
            change_dict[column] = 'str'

    df_clean = change_type(df_red, change_dict)

    # Final check
    display(check_quality(df_red))

    return df_clean

In [ ]:
df_clean = clean_df(df)

,column,type,elements,unique,rate_unique,missing,rate_missing,duplicated
0,year,<class 'numpy.float64'>,4321,21,0.49,0,0.0,4300
1,gender,<class 'str'>,4321,2,0.05,0,0.0,4319
2,age,<class 'numpy.float64'>,4321,56,1.30,0,0.0,4265
3,weight,<class 'numpy.float64'>,4321,244,5.65,0,0.0,4077
4,height,<class 'numpy.float64'>,4321,130,3.01,0,0.0,4191
5,eye_height,<class 'numpy.float64'>,4321,130,3.01,0,0.0,4191
6,shoulder_height,<class 'numpy.float64'>,4321,147,3.40,0,0.0,4174
7,elbow_height,<class 'numpy.float64'>,4321,148,3.43,0,0.0,4173
8,fingertip_height,<class 'numpy.float64'>,4321,145,3.36,0,0.0,4176
9,shoulder_width,<class 'numpy.float64'>,4321,120,2.78,0,0.0,4201


In [ ]:
# --- Basic EDA

# General use functions
def detect_outliers(data: pd.DataFrame, column: str):
    mu = data[column].mean()
    q1, q3 = data[column].quantile(0.25), data[column].quantile(0.75)
    iqr = q3 - q1

    min_limit = mu - 1.5 * iqr
    max_limit = mu + 1.5 * iqr

    outliers_index = data[(data[column] < min_limit) | (data[column] > max_limit)].index

    return outliers_index

def plot_density(data: pd.Series, column: str, hist_bins = 5):
    plt.figure(figsize = (5, 5))
    sns.histplot(data[column], kde = True, bins = hist_bins)
    plt.xlabel(column.upper())
    plt.ylabel('Density')
    plt.title(f'{column.upper()} distribution')






# General distribution

In [44]:
mu = df_clean['weight'].mean()
q1, q3 = df_clean['weight'].quantile(0.25), df_clean['weight'].quantile(0.75)
iqr = q3 - q1

min_limit = mu - 1.5 * iqr
max_limit = mu + 1.5 * iqr

outliers_index = df_clean[(df_clean['weight'] < min_limit) | (df_clean['weight'] > max_limit)].index
print(outliers_index)

Index([ 1821,  1876,  1894,  2230,  2443,  2454,  2476,  2502,  2528,  2549,
       ...
       11061, 11076, 11132, 11143, 11194, 11202, 11300, 11302, 11325, 11339],
      dtype='int64', length=132)
